## Initi fichiers et chargement

In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

import pandas as pd

df = pd.read_csv("BPE24.csv", sep=";")
# Aperçu rapide
print(df.info())

C:\Users\carine\AppData\Local\Temp\ipykernel_7880\1884733497.py:7: DtypeWarning: Columns (0: DEPCOM, 1: DEP, 2: QUALI_IRIS, 3: QUALI_QP2024, 4: QUALI_QVA, 5: QUALI_ZUS, 6: EPCI, 7: UU2020, 8: BV2022, 9: AAV2020) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("BPE24.csv", sep=";")


<class 'pandas.DataFrame'>
RangeIndex: 2827407 entries, 0 to 2827406
Data columns (total 89 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   AN                        int64  
 1   NOMRS                     str    
 2   CNOMRS                    str    
 3   NUMVOIE                   float64
 4   INDREP                    str    
 5   TYPVOIE                   str    
 6   LIBVOIE                   str    
 7   CADR                      str    
 8   CODPOS                    float64
 9   DEPCOM                    object 
 10  DEP                       object 
 11  REG                       int64  
 12  LIBCOM                    str    
 13  DOM                       str    
 14  SDOM                      str    
 15  TYPEQU                    str    
 16  SIRET                     float64
 17  STATUT_DIFFUSION          str    
 18  CANTINE                   str    
 19  INTERNAT                  str    
 20  RPI                       str    
 

In [3]:
# Dictionnaire de regroupement logique
filtres_bpe = {
    'alim_prox': ['B203', 'B204', 'B201', 'B202', 'B205'],
    'sante_prox': ['D201', 'D301', 'D307', 'D221'],
    'vie_quartier': ['A203', 'A206', 'A504', 'B312'],
    'grands_commerces': ['B104', 'B105'],
    'hopitaux': ['D101']
}

# Fonction pour extraire et nettoyer (Qualité A ou B recommandée)
def get_clean_group(df, codes):
    # On filtre les codes ET on garde une géolocalisation de qualité (A ou B)
    # A = numéro de rue, B = milieu de rue
    mask = (df['TYPEQU'].isin(codes)) & (df['QUALITE_XY'].isin(['A', 'B']))
    return df[mask][['LATITUDE', 'LONGITUDE']].dropna()

# Création des sous-groupes
df_alim = get_clean_group(df, filtres_bpe['alim_prox'])
df_sante = get_clean_group(df, filtres_bpe['sante_prox'])
df_vie = get_clean_group(df, filtres_bpe['vie_quartier'])
df_super = get_clean_group(df, filtres_bpe['grands_commerces'])
df_hosp = get_clean_group(df, filtres_bpe['hopitaux'])

print(f"Points extraits : Alim({len(df_alim)}), Santé({len(df_sante)}), Vie({len(df_vie)})")

Points extraits : Alim(48746), Santé(17981), Vie(249480)


## Chargement ficher dvf ave ecole pour ajout nouvelles colonnes services 

In [4]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Chargement du fichier prêt pour le ML
df_dvf = pd.read_csv('dvf_bpe_ecoles.csv')

print(f"Fichier ML chargé : {df_dvf.shape[0]} lignes.")

Fichier ML chargé : 4640555 lignes.


## Ajout colonnes distances services

In [5]:
from sklearn.neighbors import BallTree
import numpy as np

def calcul_dist_km(df_destination, df_source, col_name):
    """
    Calcule la distance la plus proche entre chaque bien DVF 
    et le groupe d'équipement BPE choisi.
    """
    # 1. Conversion en radians pour Haversine
    dest_rad = np.deg2rad(df_destination[['LATITUDE', 'LONGITUDE']].values)
    source_rad = np.deg2rad(df_source[['latitude', 'longitude']].values)
    
    # 2. Construction de l'arbre (BallTree)
    tree = BallTree(dest_rad, metric='haversine')
    
    # 3. Recherche du plus proche voisin
    dist, _ = tree.query(source_rad, k=1)
    
    # 4. Conversion du résultat en km (Rayon de la Terre = 6371 km)
    df_source[col_name] = dist * 6371
    return df_source

# Application du calcul pour tes 5 nouveaux piliers
print("Calcul des distances en cours...")

df_dvf = calcul_dist_km(df_alim, df_dvf, 'dist_com_prox')
df_dvf = calcul_dist_km(df_sante, df_dvf, 'dist_sante_prox')
df_dvf = calcul_dist_km(df_vie, df_dvf, 'dist_vie_quartier')
df_dvf = calcul_dist_km(df_super, df_dvf, 'dist_supermarche')
df_dvf = calcul_dist_km(df_hosp, df_dvf, 'dist_hopital')

print("Calcul terminé ! 5 nouvelles colonnes ajoutées.")

Calcul des distances en cours...
Calcul terminé ! 5 nouvelles colonnes ajoutées.


In [6]:
# Vérification rapide des nouvelles distances
cols_news = ['dist_com_prox', 'dist_sante_prox', 'dist_vie_quartier', 'dist_supermarche', 'dist_hopital']
print(df_dvf[cols_news].describe())

       dist_com_prox  dist_sante_prox  dist_vie_quartier  dist_supermarche  \
count   4.640555e+06     4.640555e+06       4.640555e+06      4.640555e+06   
mean    1.123335e+00     1.482193e+00       6.626953e-01      2.126518e+00   
std     4.871368e+00     5.071364e+00       4.724959e+00      5.539319e+00   
min     1.048206e-05     2.260135e-05       2.061161e-06      2.716456e-05   
25%     1.604056e-01     2.219235e-01       8.025777e-02      3.425938e-01   
50%     3.881040e-01     4.829166e-01       2.012493e-01      7.374524e-01   
75%     1.060118e+00     1.498818e+00       4.981881e-01      2.472136e+00   
max     2.409803e+02     2.426987e+02       2.408062e+02      2.464505e+02   

       dist_hopital  
count  4.640555e+06  
mean   6.135341e+00  
std    8.062516e+00  
min    3.611415e-03  
25%    1.208571e+00  
50%    3.120647e+00  
75%    9.536772e+00  
max    2.601750e+02  


In [7]:
df_dvf.to_csv('dvf_bpe_ecole_commerce_sante.csv', index=False)
print("Fichier sauvegardé avec succès !")

Fichier sauvegardé avec succès !
